In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/test_solution_01.csv
/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final/img_4_684.jpg
/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final/img_3_835.jpg
/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final/img_0_340.jpg
/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final/img_2_138.jpg
/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final/img_3_570.jpg
/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final/img_1_210.jpg
/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final/img_2_792.jpg
/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final/img_1_581.jpg
/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final/img_2_443.jpg
/kaggle/input/cs-460-muffin-vs-

In [10]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models, datasets
from tqdm import tqdm

# Set device to GPU if available
device = torch.device("cpu")
print(f"Using device: {device}")

# Define paths (Adjust the train_dir if your folders are structured differently)
BASE_DIR = '/kaggle/input/cs-460-muffin-vs-chihuahua-classification-challenge'
TRAIN_DIR = f'{BASE_DIR}/train' # Assuming train images are in subfolders: /train/muffin and /train/chihuahua
TEST_DIR = f'{BASE_DIR}/kaggle_test_final'

BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.001

Using device: cpu


In [14]:
# Image transformations
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load Training Data (Assumes ImageFolder structure: /muffin/ and /chihuahua/)
try:
    train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    classes = train_dataset.classes
    print(f"Classes found: {classes}")
except Exception as e:
    print("Error loading training data. Ensure your training folder has 'muffin' and 'chihuahua' subdirectories.")
    print(e)

# Custom Dataset for Test Data (since it's just a folder of images without labels)
class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.image_files = [f for f in os.listdir(test_dir) 
                    if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        return image, img_name

test_dataset = TestDataset(test_dir=TEST_DIR, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

Classes found: ['chihuahua', 'muffin']


In [12]:
# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Modify the final fully connected layer for 2 classes (Muffin vs Chihuahua)
num_ftrs = model.fc.in_features
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(num_ftrs, num_classes)

model = model.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [13]:
print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)

Classes: ['chihuahua', 'muffin']
Class mapping: {'chihuahua': 0, 'muffin': 1}


In [5]:
print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.2f}%")

Starting Training...


Epoch 1/5: 100%|██████████| 148/148 [00:40<00:00,  3.68it/s]


Epoch [1/5] - Loss: 0.1640 - Accuracy: 94.25%


Epoch 2/5: 100%|██████████| 148/148 [00:31<00:00,  4.76it/s]


Epoch [2/5] - Loss: 0.0911 - Accuracy: 96.87%


...

Epoch [3/5] - Loss: 0.0758 - Accuracy: 96.96%


Epoch 4/5: 100%|██████████| 148/148 [00:31<00:00,  4.70it/s]


Epoch [4/5] - Loss: 0.0538 - Accuracy: 97.97%


Epoch 5/5: 100%|██████████| 148/148 [00:30<00:00,  4.78it/s]

Epoch [5/5] - Loss: 0.0672 - Accuracy: 97.72%


In [6]:
print("Starting Inference on Test Set...")
model.eval()
predictions = []
filenames = []

# Map numerical outputs back to string labels
# Ensure these match the exact string format required by the competition
idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}

with torch.no_grad():
    for images, batch_filenames in tqdm(test_loader, desc="Predicting"):
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        # Convert tensor to list of integers
        predicted_classes = predicted.cpu().numpy()
        
        for idx in predicted_classes:
            predictions.append(idx_to_class[idx])
            
        filenames.extend(batch_filenames)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'ID': filenames,
    'Label': predictions # Change to 'Predict' if Kaggle throws a column name error
})

# Save to CSV
submission_df.to_csv('submission.csv', index=False)
print("Submission saved successfully as submission.csv!")
print(submission_df.head())

Starting Inference on Test Set...


Predicting: 100%|██████████| 36/36 [00:09<00:00,  3.64it/s]

Submission saved successfully as submission.csv!
              ID      Label
0  img_4_684.jpg  chihuahua
1  img_3_835.jpg     muffin
2  img_0_340.jpg  chihuahua
3  img_2_138.jpg     muffin
4  img_3_570.jpg     muffin
